### WIDS Datathon Challenge Overview:

In this year’s WiDS Datathon, participants were tasked with building a model to predict both an individual’s sex and their ADHD diagnosis using functional brain imaging data of children and adolescents and their socio-demographic, emotions, and parenting information.

The URL for the competition and its datasets is: https://www.kaggle.com/competitions/widsdatathon2025/overview.

Challenge Question and Task:
“What brain activity patterns are associated with ADHD; are they different between males and females, and, if so, how?”

The task is to create a multi-outcome model to predict two separate target variables: 1) ADHD (1=yes or 0=no) and 2) female (1=yes or 0=no).

Why is this important?
Tools of this nature can help identify individuals who may be at risk of ADHD, which can be difficult to diagnose, particularly in females. Importantly, they help shed light on the parts of the brain relevant to ADHD in females and males, which in turn could lead to improvements in personalized medicine and therapies. Identifying ADHD early and designing therapies targeting specific brain mechanisms in a personalized way can greatly improve the mental health of affected individuals.

The Python libraries used are Pandas, NumPy, Seaborn, Matplotlib, Sklearn, Scipy, and Missingno. The machine learning model I used is XGBClassifier. 

### Importing Python libraries and load the TRAIN_CATAGORICAL_METADATA_new dataframe

In [19]:
import numpy as np
import pandas as pd
import seaborn as sns

import os
import matplotlib.pyplot as plt

import sklearn
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, accuracy_score, confusion_matrix, roc_curve
from scipy.stats import zscore, pearsonr, uniform
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV

from scipy.io import loadmat

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

train_cat = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx')
train_cat.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx'

### Checking if any columns have null values

In [ ]:
print(train_cat.isna().sum())

### Checking each columns datatype.

In [ ]:
train_cat.info()

### Filling null values in each column with the rounded mean of the column

In [ ]:
train_cat.fillna({'PreInt_Demos_Fam_Child_Ethnicity':train_cat['PreInt_Demos_Fam_Child_Ethnicity'].mean().round()}, inplace = True)
train_cat.fillna({'PreInt_Demos_Fam_Child_Race':train_cat['PreInt_Demos_Fam_Child_Race'].mean().round()}, inplace = True)
train_cat.fillna({'MRI_Track_Scan_Location':train_cat['MRI_Track_Scan_Location'].mean().round()}, inplace = True)
train_cat.fillna({'Barratt_Barratt_P1_Edu':train_cat['Barratt_Barratt_P1_Edu'].mean().round()}, inplace = True)
train_cat.fillna({'Barratt_Barratt_P1_Occ':train_cat['Barratt_Barratt_P1_Occ'].mean().round()}, inplace = True)
train_cat.fillna({'Barratt_Barratt_P2_Edu':train_cat['Barratt_Barratt_P2_Edu'].mean().round()}, inplace = True)
train_cat.fillna({'Barratt_Barratt_P2_Occ':train_cat['Barratt_Barratt_P2_Occ'].mean().round()}, inplace = True)
print(train_cat.isna().sum().sum())

### Changing the datatype of each column to int to prepare for one-hot-encoding.

In [ ]:
train_cat['PreInt_Demos_Fam_Child_Ethnicity'] = train_cat['PreInt_Demos_Fam_Child_Ethnicity'].astype(int)
train_cat['PreInt_Demos_Fam_Child_Race'] = train_cat['PreInt_Demos_Fam_Child_Race'].astype(int)
train_cat['MRI_Track_Scan_Location'] = train_cat['MRI_Track_Scan_Location'].astype(int)
train_cat['Barratt_Barratt_P1_Edu'] = train_cat['Barratt_Barratt_P1_Edu'].astype(int)
train_cat['Barratt_Barratt_P1_Occ'] = train_cat['Barratt_Barratt_P1_Occ'].astype(int)
train_cat['Barratt_Barratt_P2_Edu'] = train_cat['Barratt_Barratt_P2_Edu'].astype(int)
train_cat['Barratt_Barratt_P2_Occ'] = train_cat['Barratt_Barratt_P2_Occ'].astype(int)

In [ ]:
for col in train_cat.select_dtypes(include='int').columns:
    train_cat[col] = train_cat[col].astype('category')

### Creating a list of all of the columns except the first (Participant id).

In [ ]:
# Creating a list of all of the columns except the first (Participant id)
columns_to_encode = train_cat.columns[1:].tolist()

# Print the columns to encode
print("Columns to encode:", columns_to_encode)

### One-hot-encoding the categorical data

In [ ]:
# encoding categorical data
train_encoded = pd.get_dummies(train_cat[columns_to_encode], drop_first=True)
train_encoded = train_encoded.applymap(lambda x: 1 if x is True else (0 if x is False else x))

In [ ]:
# Combine encoded columns with the rest of the DataFrame
cat_train_final = pd.concat([train_cat.drop(columns=columns_to_encode), train_encoded], axis=1)

In [ ]:
# ensure it looks correct
cat_train_final.head()

In [ ]:
# load train_solutions dataframe
train_solutions = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx')
train_solutions.head()

### Dropping the 'participant_id' column from cat_train_final and train_solutions datasets because it is irrelevant for feature selection.

In [ ]:
X = cat_train_final.drop(columns = ['participant_id'])
y = train_solutions.drop(columns = ['participant_id'])

In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

# Initialize the base classifier
xgb_classifier = XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=5)

In [ ]:
# Wrap with MultiOutputClassifier for multi-target classification
multioutput_classifier = MultiOutputClassifier(xgb_classifier)

In [ ]:
# Assuming cat_train_final is a DataFrame
X = cat_train_final.drop(columns = ['participant_id'], axis=1)  
y = train_solutions.drop(columns = ['participant_id'])
y = train_solutions['Sex_F']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = xgb.XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

# Get feature importance
feature_importance = model.feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print(importance_df)

# Optional: Plot feature importance

# Plot using Seaborn
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
import pandas as pd

# Assuming cat_train_final is a DataFrame
X = cat_train_final.drop(columns = ['participant_id'], axis=1)  # Replace 'target' with your target column name
y = train_solutions.drop(columns = ['participant_id'])
y = train_solutions['ADHD_Outcome']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = xgb.XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

# Get feature importance
feature_importance = model.feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print(importance_df)

# Optional: Plot feature importance

# Plot using Seaborn
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
# Load quantitative metadata
train_Quant = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx')
print("Quantitative Data:")
print(train_Quant.head())

In [ ]:
# Functional Connectome Matrices

file_path_trainFCM = "Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"
train_FCM = pd.read_csv(file_path_trainFCM)
train_FCM.head()

In [ ]:
train_FCM.columns

In [ ]:
train_Quant.columns

### The quant values

In [ ]:
# Load categorical metadata
train_quant_explore = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx')
print("\Quant Data:")
print(train_quant_explore.head())


# Print number of missing values for each column
print("\nMissing values per column:")
print(train_quant_explore.isnull().sum())

In [ ]:
# create a new dataframe with the columns that have missing values
missing_columns = train_quant_explore.columns[train_quant_explore.isnull().any()]
train_quant_explore_missing = train_quant_explore[missing_columns]
# drop the MRI_Track_Age_at_Scan column from the dataframe
train_quant_explore_missing = train_quant_explore_missing.drop(columns=['MRI_Track_Age_at_Scan'])

# Summarize the missing values in the new dataframe
print("\nMissing values in the new dataframe:")
print(train_quant_explore_missing.isnull().sum())
# Print the total nuumber of rows with missing values
print("\nTotal number of rows with missing values:")
print(train_quant_explore_missing.isnull().sum().sum())

In [ ]:
!pip install missingno

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno

fig, axes = plt.subplots(1, 2, figsize=(18, 9))  # 1 row, 2 columns

# Plot matrix with smaller font
msno.matrix(train_quant_explore, ax=axes[0])
axes[0].set_title('Missing Value Matrix', fontsize=12)

# Plot heatmap
msno.heatmap(train_quant_explore, ax=axes[1])
axes[1].set_title('Missingness Heatmap', fontsize=12)


# Reduce tick label font sizes
for i in [0,1]:
    for label in axes[i].get_xticklabels():
        label.set_fontsize(8)
    for label in axes[i].get_yticklabels():
        label.set_fontsize(10)

plt.tight_layout()
plt.show()

In [ ]:
# Count missing values per row
train_quant_explore['num_missing'] = train_quant_explore.isnull().sum(axis=1)
# See rows with multiple missing values
print(train_quant_explore[train_quant_explore['num_missing'] > 1])

In [ ]:
# the box plot for columns with missing values
plt.figure(figsize=(12, 6))
missing_columns = train_quant_explore.columns[train_quant_explore.isnull().any()]
sns.boxplot(data=train_quant_explore[missing_columns], orient='h')
plt.title('Boxplot of Columns with Missing Values')
plt.xlabel('Values')
plt.ylabel('Columns')
plt.show()
# the box plot for columns with missing values
# for each of the columns with missing values, summarize in the table the number of missing values, mean, median, and standard deviation of the values in the column
missing_summary = train_quant_explore[missing_columns].describe().T[['mean', '50%', 'std', 'count']]
missing_summary.columns = ['mean', 'median', 'std', 'count']
missing_summary['missing'] = train_quant_explore[missing_columns].isnull().sum()
missing_summary = missing_summary.drop(columns=['count'] )
print(missing_summary)

In [ ]:
train_quant_explore

In [ ]:
# impute with mean for the quant dataset
# Make a copy
train_quant_explore_imputed = train_quant_explore.copy()

# Select only numeric columns
numeric_cols = train_quant_explore_imputed.select_dtypes(include=['number']).columns

# Fill missing values in numeric columns with the mean
train_quant_explore_imputed[numeric_cols] = train_quant_explore_imputed[numeric_cols].fillna(
    train_quant_explore_imputed[numeric_cols].mean()
)

# Check if any missing values remain
print("\nMissing values in the imputed data:")
print(train_quant_explore_imputed.isnull().sum().sum())

In [ ]:
# summarize the imputed data
print("\nImputed Data Summary:")
print(train_quant_explore_imputed.describe())

# print the bar plot for the imputed data
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_quant_explore_imputed[numeric_cols], orient='h')
plt.title('Boxplot of Mean Imputed Columns')
plt.xlabel('Values')
plt.ylabel('Columns')
plt.show()

In [ ]:
# impute with the knn regression for the quant dataset
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Make a copy
train_quant_explore_knn_imputed = train_quant_explore.copy()
# Select only numeric columns
numeric_cols = train_quant_explore_knn_imputed.select_dtypes(include=['number']).columns
# Select only the columns with missing values
missing_cols = train_quant_explore_knn_imputed.columns[train_quant_explore_knn_imputed.isnull().any()]
# Create a KNN imputer
imputer = KNNImputer(n_neighbors=5, weights='uniform')
# Fit the imputer on the data
imputer.fit(train_quant_explore_knn_imputed[numeric_cols])
# Transform the data
train_quant_explore_knn_imputed[numeric_cols] = imputer.transform(train_quant_explore_knn_imputed[numeric_cols])
# Check if any missing values remain
print("\nMissing values in the imputed data:")
print(train_quant_explore_knn_imputed.isnull().sum().sum())

In [ ]:
# summarize the imputed data
print("\nImputed Data Summary:")
print(train_quant_explore_knn_imputed.describe())


# print the bar plot for the imputed data
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_quant_explore_knn_imputed[numeric_cols], orient='h')
plt.title('Boxplot of KNN Imputed Columns')
plt.xlabel('Values')
plt.ylabel('Columns')
plt.show()

In [ ]:
train_quant_explore_imputed.select_dtypes(include=['number']).columns

In [ ]:
train_quant_explore_imputed

In [ ]:
train_cat_FCM = pd.merge(cat_train_final, train_FCM, on = 'participant_id')

In [ ]:
# impute with mean for the quant dataset
# Make a copy
train_quant = train_quant_explore.copy()

# Select only numeric columns
numeric_cols = train_Quant.select_dtypes(include=['number']).columns

# Fill missing values in numeric columns with the mean
train_Quant[numeric_cols] = train_Quant[numeric_cols].fillna(
    train_Quant[numeric_cols].mean()
)

# Check if any missing values remain
print("\nMissing values in the imputed data:")
print(train_Quant.isnull().sum().sum())

In [ ]:
train_df = pd.merge(train_cat_FCM, train_Quant, on = 'participant_id')

# ensure it looks accurate
train_df.head()

In [ ]:
# check how many NA values we have
print(train_df.isna().sum())

In [ ]:
participant_id = train_df['participant_id']
X = train_df.drop(columns = ['participant_id'])
y = train_solutions.drop(columns = ['participant_id'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

# Initialize the base classifier
xgb_classifier = XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=5)

In [ ]:
# Wrap with MultiOutputClassifier for multi-target classification
multioutput_classifier = MultiOutputClassifier(xgb_classifier)

In [ ]:
# Train the model
multioutput_classifier.fit(X, y)

In [ ]:
multioutput_classifier.score(X_train, y_train)

In [ ]:
y_pred = multioutput_classifier.predict(X_test)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Evaluate the F1 score
f1 = f1_score(y_test, y_pred, average='micro')  # You can choose 'micro', 'macro', or 'weighted'
print("F1 Score:", f1)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, accuracy_score

In [ ]:
def multi_output_accuracy(y_true, y_pred):
    # Ensure y_true and y_pred are NumPy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Compute accuracy for each target variable and return the mean
    return np.mean([accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])])

In [ ]:
# Create a scorer using scikit-learn's make_scorer
multi_output_scorer = make_scorer(multi_output_accuracy)

In [ ]:
#Note: it takes 12 min to run this cell!
# Perform cross-validation on the training data
cv_scores = cross_val_score(multioutput_classifier, X_train, y_train, cv=5, scoring=multi_output_scorer)

# Output the cross-validation results
print("Cross-validation scores for each fold:", cv_scores)
print("Mean CV score:", np.mean(cv_scores))

# Cross-validation scores for each fold: [0.82304527 0.78600823 0.69341564 0.64669421 0.33471074]
# Mean CV score: 0.6567748188960311

In [ ]:
from sklearn.linear_model import LogisticRegression

### 2nd Iteration

In [21]:
# Load quantitative metadata
train_Quant = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx')
print("Quantitative Data:")
print(train_Quant.head())

# Load categorical metadata
train_cat = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx')
print("Categorical Data:")
print(train_cat.head())

# Load training solutions
train_Solutions = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx')
print("\nTraining Solutions:")
print(train_Solutions.head())


file_path_trainFCM = "Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"
train_FCM = pd.read_csv(file_path_trainFCM)
train_FCM.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Documents/KStateSpring2025/widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx'

In [ ]:
# Load test quantitative metadata
test_Quant = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TEST/TEST_QUANTITATIVE_METADATA.xlsx')
print("\nTest Quantitative Data:")
print(test_Quant.head())


# Load test categorical metadata
test_cat = pd.read_excel('Documents/KStateSpring2025/widsdatathon2025/TEST/TEST_CATEGORICAL.xlsx')
print("\nTest Categorical Data:")
print(test_cat.head())


# Load test FCM data
file_path_testFCM = "Documents/KStateSpring2025/widsdatathon2025/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv"
test_FCM = pd.read_csv(file_path_testFCM)
test_FCM.head()

In [ ]:
#Checking null values in categorical dataframe
train_cat.isnull().sum()

In [ ]:
train_cat.head()

### One-hot encoding categorical dataframe

In [ ]:
def one_hot_encode_categoricals(train_cat: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode all categorical columns (int, object, bool, category) 
    except the first column. Includes NaN in the encoding if present.
    """

    # Columns to encode (excluding first column)
    columns_to_encode = train_cat.columns[1:]

    # Filter to include only intended categorical columns

    # Convert to 'category' dtype
    for col in columns_to_encode:
        train_cat[col] = train_cat[col].astype('category')
        # no need to manually add NaN as category — get_dummies(dummy_na=True) will handle it

    # One-hot encode with NaN included as a separate column
    train_encoded = pd.get_dummies(
        train_cat[columns_to_encode], drop_first=True, dummy_na=True, dtype='int8')

    # Combine the unencoded first column with the encoded columns
    cat_train_final = pd.concat(
        [train_cat.iloc[:, [0]], train_encoded], axis=1)

    return cat_train_final

In [ ]:
# Demonstration of the function
print('The columns in the pre-processed categorical metadata are:',
      train_cat.columns.tolist())
print(train_cat.info())

In [ ]:
# One-hot encode the categorical columns
train_cat_exp = train_cat.copy()
train_cat_exp = one_hot_encode_categoricals(train_cat_exp)
print('The columns in the categorical metadata after one-hot encoding are:',
      train_cat_exp.columns)
print(train_cat_exp.info())

In [ ]:
# imputation v0: imput the missing values with mean and mode
# the missing values in the train_cat dataset are categorical, so we can use the mode to fill them
train_cat_im_v0 = train_cat.copy()
train_cat_im_v0.fillna(train_cat_im_v0.mode().iloc[0], inplace=True)
train_cat_im_v0 = one_hot_encode_categoricals(train_cat_im_v0)

# the missing values in the train_Quant dataset are numerical, so we can use the mean to fill them
train_Quant_im_v0 = train_Quant.copy()
train_Quant_im_v0 = train_Quant_im_v0.apply(lambda col: col.fillna(col.mean()) if col.dtype in ['float64', 'int64'] else col)

# standardize the numerical features
scaler = StandardScaler()
train_Quant_im_v0.iloc[:, 1:] = scaler.fit_transform(
    train_Quant_im_v0.iloc[:, 1:])
train_FCM_scaled = train_FCM.copy()
# standardize the functional connectome matrices
train_FCM_scaled.iloc[:, 1:] = scaler.fit_transform(
    train_FCM_scaled.iloc[:, 1:])

In [ ]:
train_cat_im_v0.isnull().sum()

In [ ]:
train_Quant_im_v0.isnull().sum()

### Feature Selection for Categorical dataframe target ADHD_Outcome

In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

# Initialize the base classifier
xgb_classifier = XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=5)

In [ ]:
# Wrap with MultiOutputClassifier for multi-target classification
multioutput_classifier = MultiOutputClassifier(xgb_classifier)

In [ ]:
X = train_cat_im_v0.drop(columns = ['participant_id'], axis=1)  
y = train_Solutions['ADHD_Outcome']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

selector = XGBClassifier(X_train, threshold='mean', prefit=True)
# Get feature importance
feature_importance = model.feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print(importance_df)

# Optional: Plot feature importance

# Plot using Seaborn
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier

In [ ]:
X = train_cat_im_v0.drop(columns = ['participant_id'], axis=1)  
y = train_Solutions['ADHD_Outcome']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

# Step 2: Use SelectFromModel for Feature Selection
# Select features with importance greater than the mean importance
selector = SelectFromModel(model, threshold='mean', prefit=True)
# Get feature importance

# Transform the dataset to keep only the selected features
train_cat_selected = selector.transform(X)

# Step 3: Get the selected feature names
selected_features_ADHD_list = X.columns[selector.get_support()]

# Output the results
print("Original number of features:", X.shape[1])
print("Number of selected features:", train_cat_selected.shape[1])
print("Selected features:", selected_features_ADHD_list.tolist())

In [ ]:
train_cat_selected.shape

In [ ]:
train_cat_final = train_cat_selected.copy()
train_cat_final

### Feature Selection for Quantitative dataframe target Sex_F

In [ ]:
X = train_Quant_im_v0.drop(columns = ['participant_id'], axis=1)  
y = train_Solutions['Sex_F']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

selector = SelectFromModel(model, threshold='mean', prefit=True)
# Get feature importance
feature_importance = model.feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print(importance_df)

# Optional: Plot feature importance

# Plot using Seaborn
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
X = train_Quant_im_v0.drop(columns = ['participant_id'], axis=1)  
y = train_Solutions['Sex_F']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
model = XGBClassifier(eval_metric='logloss')  # Use XGBRegressor for regression tasks
model.fit(X_train, y_train)

# Step 2: Use SelectFromModel for Feature Selection
# Select features with importance greater than the mean importance
selector = SelectFromModel(model, threshold='mean', prefit=True)
# Get feature importance

# Transform the dataset to keep only the selected features
train_Quant_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# Step 3: Get the selected feature names
selected_features_SexF_list = X.columns[selector.get_support()]

# Output the results
print("Original number of features:", X.shape[1])
print("Number of selected features:", train_Quant_selected.shape[1])
print("Selected features:", selected_features_SexF_list.tolist())

In [ ]:
train_Quant_final = train_Quant_selected.copy()

In [ ]:
# converting train_cat_final and train_Quant_final from numpy arrays to DataFrames
train_cat_final = pd.DataFrame(train_cat_final, columns=selected_features_ADHD_list)  # Replace with actual column names
train_Quant_final = pd.DataFrame(train_Quant_final, columns=selected_features_SexF_list)  # Replace with actual column names

# Display the converted DataFrames
print(train_cat_final)
print(train_Quant_final)

In [ ]:
print(train_cat_final.shape)
print(train_Quant_final.shape)

In [ ]:
# Putting back the 'participant_id' column
train_cat_final = pd.concat([train_cat_im_v0[['participant_id']], train_cat_final], axis=1)
train_Quant_final= pd.concat([train_Quant_im_v0[['participant_id']], train_Quant_final], axis=1)

### Merging train dataframes

In [ ]:
train_cat_FCM = pd.merge(train_cat_final, train_FCM, on = 'participant_id')

In [ ]:
train_cat_FCM.head()

In [ ]:
train_df = pd.merge(train_cat_FCM, train_Quant_final, on = 'participant_id')

# ensure it looks accurate
train_df.head()
train_df.columns

In [ ]:
train_df = train_df.apply(lambda col: col.fillna(col.mean()) if col.dtype in ['float64', 'int64'] else col)
# check how many NA values we have
print(train_df.isnull().sum())

In [ ]:
participant_id = train_df['participant_id']
X = train_df.drop(columns = ['participant_id'])
y = train_Solutions.drop(columns = ['participant_id'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Initialize the base classifier
xgb_classifier = XGBClassifier(objective='binary:logistic', n_estimators=500, learning_rate=0.1, max_depth=5)

In [ ]:
# Wrap with MultiOutputClassifier for multi-target classification
multioutput_classifier = MultiOutputClassifier(xgb_classifier)

In [ ]:
# Train the model
multioutput_classifier.fit(X, y)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, accuracy_score

In [ ]:
def multi_output_accuracy(y_true, y_pred):
    # Ensure y_true and y_pred are NumPy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Compute accuracy for each target variable and return the mean
    return np.mean([accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])])

In [ ]:
# Create a scorer using scikit-learn's make_scorer
multi_output_scorer = make_scorer(multi_output_accuracy)

In [ ]:
#Note: it takes 12 min to run this cell!
# Perform cross-validation on the training data
cv_scores = cross_val_score(multioutput_classifier, X_train, y_train, cv=5, scoring=multi_output_scorer)

# Output the cross-validation results
print("Cross-validation scores for each fold:", cv_scores)
print("Mean CV score:", np.mean(cv_scores))

### Prep Test Data

In [ ]:
# imputation v0: imput the missing values with mean and mode
# the missing values in the test_cat dataset are categorical, so we can use the mode to fill them
test_cat_im_v0 = test_cat.copy()
test_cat_im_v0.fillna(test_cat_im_v0.mode().iloc[0], inplace=True)
test_cat_im_v0 = one_hot_encode_categoricals(test_cat_im_v0)

'''
# for columns that exist in train_cat but not in test_cat, we need to add them with NaN values
missing_cols = set(train_cat_final.columns) - set(test_cat_im_v0.columns)
for col in missing_cols:
    test_cat_im_v0[col] = 0  # use 0 instead of np.nan

# Drop columns to match training data
#test_cat_im_v0 = test_cat_im_v0.drop(columns = ['Barratt_Barratt_P1_Occ_nan', 'PreInt_Demos_Fam_Child_Race_7.0', 'Barratt_Barratt_P1_Occ_15.0', 'Basic_Demos_Enroll_Year_2020', 'Barratt_Barratt_P1_Edu_nan', 'Basic_Demos_Enroll_Year_nan', 'Barratt_Barratt_P2_Edu_9.0', 'Barratt_Barratt_P2_Occ_10.0', 'Barratt_Barratt_P2_Occ_15.0', 'Barratt_Barratt_P1_Edu_9.0', 'MRI_Track_Scan_Location_nan', 'Barratt_Barratt_P2_Edu_6.0', 'Barratt_Barratt_P2_Edu_nan', 'PreInt_Demos_Fam_Child_Ethnicity_nan', 'PreInt_Demos_Fam_Child_Race_9.0', 'Barratt_Barratt_P2_Occ_nan', 'PreInt_Demos_Fam_Child_Race_11.0', 'Basic_Demos_Study_Site_nan', 'Barratt_Barratt_P2_Occ_20.0', 'Barratt_Barratt_P1_Edu_6.0', 'PreInt_Demos_Fam_Child_Ethnicity_3.0', 'PreInt_Demos_Fam_Child_Race_nan'])

# Reorder columns to match training data
test_cat_im_v0 = test_cat_im_v0[train_cat_final.columns]
'''

train_cols = set(train_cat_im_v0.columns)
test_cols = set(test_cat_im_v0.columns)

missing_in_test = train_cols - test_cols
extra_in_test = test_cols - train_cols

for col in missing_in_test:
    test_cat_im_v0[col] = 0 

test_cat_im_v0 = test_cat_im_v0[train_cat_im_v0.columns]

# the missing values in the test_Quant dataset are numerical, so we can use the mean to fill them
test_Quant_im_v0 = test_Quant.copy()
test_Quant_im_v0 = test_Quant_im_v0.apply(
    lambda col: col.fillna(col.mean()) if col.dtype in [
        'float64', 'int64'] else col
)

# drop columns to match training data
#test_Quant_im_v0 = test_Quant_im_v0.drop(columns = ['ColorVision_CV_Score', 'APQ_P_APQ_P_PM', 'APQ_P_APQ_P_ID', 'SDQ_SDQ_Prosocial', 'APQ_P_APQ_P_PP', 'EHQ_EHQ_Total', 'APQ_P_APQ_P_INV', 'SDQ_SDQ_Internalizing'])

# Reorder columns to match training data
test_Quant_im_v0 = test_Quant_im_v0[train_Quant_final.columns]

# standardize the numerical features
scaler = StandardScaler()
test_Quant_im_v0.iloc[:, 1:] = scaler.fit_transform(
    test_Quant_im_v0.iloc[:, 1:])
test_FCM_scaled = test_FCM.copy()
# standardize the functional connectome matrices
test_FCM_scaled.iloc[:, 1:] = scaler.fit_transform(test_FCM_scaled.iloc[:, 1:])


# merge the imputed datasets
test_merged_v0 = pd.merge(
    test_cat_im_v0, test_Quant_im_v0, on='participant_id')
# merge with the functional connectome matrices
test_merged_v0 = pd.merge(test_merged_v0, test_FCM_scaled, on='participant_id')
# check the missing values in the merged dataset
# Print number of missing values for each column
print("\nMissing values per column after imputation:")
print(test_merged_v0.isnull().sum())

In [ ]:
# compare the shape of the train and test datasets, check every dataset: test_cat_im_v0,test_Quant_im_v0, test_FCM_scaled
print("\nShape of the train_merged_v0 dataset:")
print(train_df.shape)
print("\nShape of the test_merged_v0 dataset:")
print(test_merged_v0.shape)

# compare the shape of every dataset: test_cat_im_v0,test_Quant_im_v0, test_FCM_scaled
print("\nShape of the train_cat_im_v0 dataset:")
print(train_cat_im_v0.shape)

print("\nShape of the test_cat_im_v0 dataset:")
print(test_cat_im_v0.shape)

# also check those in the train set

print("\nShape of the train_Quant_im_v0 dataset:")
print(train_Quant_im_v0.shape)

print("\nShape of the test_Quant_im_v0 dataset:")
print(test_Quant_im_v0.shape)

print("\nShape of the train_FCM_scaled dataset:")
print(train_FCM_scaled.shape)

print("\nShape of the test_FCM_scaled dataset:")
print(test_FCM_scaled.shape)

In [ ]:
train_cols = set(train_df.columns)
test_cols = set(test_merged_v0.columns)

missing_in_test = train_cols - test_cols
extra_in_test = test_cols - train_cols

# Adding missing columns
for col in missing_in_test:
    test_merged_v0[col] = 0 

# Reorder columns to match training data
test_merged_v0 = test_merged_v0[train_df.columns]

In [ ]:
print("Missing in test:", missing_in_test)
print("Unexpected in test:", extra_in_test)

In [ ]:
participant_id = test_merged_v0['participant_id']

X_test = test_merged_v0.drop(columns = 'participant_id')

y_pred = multioutput_classifier.predict(X_test)

In [ ]:
# Convert predictions to a DataFrame
predictions_df = pd.DataFrame(
    y_pred,
    columns=['Predicted_Gender', 'Predicted_ADHD']
)

# Combine participant IDs with predictions
result_df = pd.concat([participant_id.reset_index(drop=True), predictions_df], axis=1)

# Print or save the DataFrame
print(result_df)

In [ ]:
# Save result_df to a CSV file
result_df.to_csv('submission_5.csv', index=False)

## Conclusion

In the first iteration, I looked at the null values in the train_categorical and train_quantitative datasets, then imputed the mode for the train_categorical and the mean for the train_quantitative. The train_fcm dataset had no null values. After merging the three datasets, I trained the model, did a train-test split, and used cross-validation to give me a CV score of 0.6417. On my second iteration, I used XGBClassifier for feature selection for the train_categorical and train_quantitative datasets. Again, I imputed train_categorical with mode and train_quantitative with mean. My target variable for the train_categorical was ADHD_Outcome, and my target variable for train_quantitative was Sex_f. After training the model, I did a train-test split and cross-validation and got a CV score of 0.7536. I thought that was a significant improvement from my first iteration. I then prepped my test data and made a prediction dataframe for submission, even though I missed the deadline due to difficulty with column discrepancies between my merged train and test datasets.